# The LangGraph Pipeline — Multi-Agent Orchestration with Human Approval

This is the heart of the project. Four AI agents, a state machine that runs them, parallel execution, and a real pause-and-wait gate for high-risk decisions. This notebook builds all of it and tests it end to end.

## Why this needs a graph, not a simple chain

Most AI pipelines could be a simple sequence of function calls. The difference shows up in two specific situations:

**1. Things that can run at the same time.** The SQL agent (database lookup) and the policy agent (document search) don't depend on each other. Running them in parallel means the total wait is the slower of the two, not the sum of both. A linear chain can't do this without threading boilerplate.

**2. Pausing and resuming execution.** The human approval gate needs to genuinely stop the pipeline, save its state somewhere persistent, and wait. Later, a human reviews the draft and approves or denies it. The pipeline resumes exactly where it paused, even if the Python process was restarted in between.

LangGraph's `interrupt()` handles this through its checkpointer mechanism. When interrupt is called, the graph serializes the complete state to the checkpointer (MemorySaver here; Postgres in production). The call returns immediately — nothing is blocked. Later, you call `graph.invoke(Command(resume=...))` with the same thread ID and the graph picks up where it left off.

## What each agent does

**Triage** reads the support agent's query and figures out what evidence is needed. Does this need a database lookup? Policy documents? Both? It extracts the ticket reference and sets flags that control which branches run in parallel. This is classification and routing — a different job from any other agent.

**SQL Agent** generates the database query for ticket lookup. It doesn't execute raw SQL from the AI. It generates a query structure, and the actual database call uses parameterized values — the ticket ID goes in as a bound parameter, not interpolated into a string. Even if the AI generates something unexpected, Postgres RLS from Notebook 01 is the final safety net.

**Policy Agent** searches the in-memory policy snippets using keyword overlap. In production this calls the Qdrant hybrid search from Notebook 02. No AI call here — it's purely deterministic retrieval.

**Synthesizer** receives the merged state after both parallel branches finish. It writes the resolution email using only the evidence that was actually retrieved. The prompt is explicit about not inventing policy clauses or making promises beyond what the evidence supports.

**Risk Assessor** evaluates whether the draft needs a human to sign off. If the refund is over $100, or if the policy application is ambiguous, it flags the case as high risk. The graph then routes to the interrupt node instead of finalizing automatically.

In [1]:
# Setup: LLM client and environment.

import os
import json
import uuid
import pathlib
from typing import Optional
from dotenv import load_dotenv

project_root = pathlib.Path(".").resolve().parent
load_dotenv(project_root / ".env")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

LLM_MODEL = "qwen/qwen3.8-27b"

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not set.")

from openai import OpenAI

llm_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")


def call_llm(prompt, max_tokens=700):
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0,
    )
    return response.choices[0].message.content


def call_llm_json(prompt, max_tokens=512):
    raw = call_llm(prompt, max_tokens=max_tokens)
    cleaned = raw.strip()
    for fence in ("```json", "```"):
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"error": "json_parse_failure", "raw": raw[:300]}


print(f"LLM client ready. model={LLM_MODEL!r}")

LLM client ready. model='qwen/qwen3.8-27b', url='https://api.groq.com/openai/v1'


In [2]:
# Define the AgentState schema.
#
# This TypedDict is the shared state passed between every node.
# LangGraph merges each node's return values back into the state dict
# automatically, so each node only needs to return the fields it changed.
#
# The messages field uses Annotated with add_messages, which tells LangGraph
# to append new messages rather than replacing the whole list.

from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    query: str
    token: str
    role: str
    user_id: str
    token_valid: bool
    # Triage sets these to decide which branches run in parallel
    needs_sql: bool
    needs_policy: bool
    ticket_ref: Optional[str]
    # Evidence collected by the parallel branches
    sql_result: dict
    policy_results: list
    # Written by the synthesizer
    draft_email: str
    # Written by the risk assessor and HITL gate
    risk: dict
    approved: bool
    final_status: str
    thread_id: str


print("AgentState defined.")
print(f"Fields: {list(AgentState.__annotations__.keys())}")

AgentState defined.
Fields: ['messages', 'query', 'token', 'role', 'user_id', 'token_valid', 'needs_sql', 'needs_policy', 'ticket_ref', 'sql_result', 'policy_results', 'draft_email', 'risk', 'approved', 'final_status', 'thread_id']


In [3]:
# Build the four agent nodes.
#
# We use an in-memory mock ticket database here for portability.
# In production this is replaced by the PostgreSQL connection from Notebook 01
# with RLS session variables set from the JWT.

from langchain_core.messages import HumanMessage, AIMessage

# Small in-memory policy corpus for this notebook.
# In production the policy agent calls Qdrant (Notebook 02).
POLICY_SNIPPETS = [
    "Standard return window: 30 days for general merchandise, 15 days for electronics.",
    "Approved refunds processed within 5-7 business days to original payment method.",
    "Shipping damage must be reported within 48 hours with photographic evidence.",
    "Goodwill refunds up to $150 may be approved by supervisors without VP sign-off.",
    "Customers with 3 or more refund requests in 90 days are flagged for abuse review.",
    "Items in original unused condition qualify for a full refund.",
    "Items showing signs of use may receive a partial refund of up to 50%.",
]


def simple_policy_search(query, top_k=3):
    """Keyword overlap search — a stand-in for Qdrant hybrid search."""
    query_words = set(query.lower().split())
    scored = [(s, len(query_words & set(s.lower().split()))) for s in POLICY_SNIPPETS]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [s for s, score in scored[:top_k] if score > 0]


# Mock ticket database. In production this is Postgres with RLS enforced.
MOCK_TICKETS = {
    "T1001": {"id": "T1001", "customer_name": "Alice Smith",
              "region": "us-east", "support_agent_id": "dave",
              "disputed_balance": 45.00, "ticket_status": "open",
              "issue_summary": "Refund not received after 30 days"},
    "T1005": {"id": "T1005", "customer_name": "Bob Johnson",
              "region": "us-west", "support_agent_id": "dave",
              "disputed_balance": 150.00, "ticket_status": "pending",
              "issue_summary": "Wrong item shipped, wants full refund"},
    "T1010": {"id": "T1010", "customer_name": "Carol White",
              "region": "eu-west", "support_agent_id": "dave",
              "disputed_balance": 275.00, "ticket_status": "escalated",
              "issue_summary": "Damaged goods, subscription charge after cancellation"},
}


def parameterized_ticket_lookup(ticket_ref, agent_id):
    """Look up a ticket using parameterized values, not string interpolation.

    The ticket_ref comes from the AI (triage agent), but it's used only as
    a lookup key in a dict — never embedded into a SQL string. In production
    this is a parameterized query with values bound separately.
    """
    if not ticket_ref:
        # No specific ticket mentioned, return all tickets for this agent
        results = [t for t in MOCK_TICKETS.values() if t["support_agent_id"] == agent_id]
        return {"rows": results, "count": len(results)}

    ticket = MOCK_TICKETS.get(ticket_ref)
    if ticket and ticket["support_agent_id"] == agent_id:
        return {"rows": [ticket], "count": 1}
    elif ticket and ticket["support_agent_id"] != agent_id:
        # Postgres RLS would block this at the database level.
        # We replicate that behavior here.
        return {"rows": [], "count": 0, "rls_blocked": True}
    return {"rows": [], "count": 0, "not_found": True}


# ---- Agent node functions ----

def triage_agent(state):
    """Agent 1: figure out what evidence is needed to answer this query.

    This is purely a classification job. It decides whether the next step
    needs database data, policy documents, or both. The result determines
    which branches run in parallel after this node.
    """
    prompt = f"""A support agent sent this message:
"{state['query']}"

Extract:
1. ticket_ref: The ticket ID (like T1001) if mentioned, otherwise null.
2. needs_sql: true if we need to look up ticket or order data from the database.
3. needs_policy: true if we need to check refund or return policy documents.

Return only valid JSON:
{{"ticket_ref": "T1001", "needs_sql": true, "needs_policy": true}}"""

    result = call_llm_json(prompt)
    return {
        "ticket_ref": result.get("ticket_ref"),
        "needs_sql": bool(result.get("needs_sql", True)),
        "needs_policy": bool(result.get("needs_policy", False)),
        "messages": [AIMessage(content=f"Triage complete: {result}")],
    }


def sql_agent(state):
    """Agent 2: look up ticket data from the database.

    The AI determines what kind of query to run. The actual database call
    uses the extracted ticket_ref as a bound parameter, not as a raw string
    embedded in SQL. Postgres RLS is the authoritative enforcement layer
    regardless of what this agent generates.
    """
    if not state.get("needs_sql"):
        return {"sql_result": {}, "messages": [AIMessage(content="SQL: skipped")]}

    ticket_ref = state.get("ticket_ref")
    agent_id = state.get("user_id", "dave")

    prompt = f"""For this support query, what database lookup is needed?
Query: {state['query']}
Ticket reference: {ticket_ref}
Agent ID: {agent_id}

Return JSON: {{"query_type": "single_ticket" or "all_agent_tickets", "ticket_id": "{ticket_ref}"}}"""

    plan = call_llm_json(prompt)
    sql_result = parameterized_ticket_lookup(
        ticket_ref=plan.get("ticket_id") or ticket_ref,
        agent_id=agent_id,
    )

    return {
        "sql_result": sql_result,
        "messages": [AIMessage(content=f"SQL: found {sql_result['count']} ticket(s)")],
    }


def policy_agent(state):
    """Policy retrieval: no AI call, just keyword search.

    This node runs in parallel with the SQL agent. In production it calls
    the Qdrant hybrid search from Notebook 02.
    """
    if not state.get("needs_policy"):
        return {"policy_results": [], "messages": [AIMessage(content="Policy: skipped")]}

    results = simple_policy_search(state["query"], top_k=3)
    return {
        "policy_results": results,
        "messages": [AIMessage(content=f"Policy: retrieved {len(results)} clauses")],
    }


def synthesizer_agent(state):
    """Agent 3: merge the evidence and write the resolution email.

    This is the fan-in node. It runs after both parallel branches finish.
    It has access to the ticket data from the SQL agent and the policy
    clauses from the policy agent in the same state dict.
    The prompt is strict about not going beyond what the evidence says.
    """
    context_parts = []

    sql = state.get("sql_result", {})
    if sql and sql.get("rows"):
        context_parts.append(f"[Ticket data]\n{json.dumps(sql['rows'], indent=2)}")

    for clause in state.get("policy_results", []):
        context_parts.append(f"[Policy clause]\n{clause}")

    if not context_parts:
        context_parts.append("No evidence found.")

    evidence = "\n\n".join(context_parts)

    prompt = f"""Write a short professional resolution email to the customer.
Use only the evidence provided below. Cite the ticket status and the policy basis.
If the evidence does not support a clear answer, say so honestly.
Keep the email under 150 words.

Evidence:
{evidence}

Support agent request: {state['query']}

Email:"""

    draft = call_llm(prompt, max_tokens=400)
    return {
        "draft_email": draft,
        "messages": [AIMessage(content=f"Synthesizer: email drafted ({len(draft)} chars)")],
    }


print("All four agent nodes defined.")

All four agent nodes defined.


In [4]:
# Risk assessor and the HITL interrupt gate.
#
# The risk assessor evaluates the draft from a different angle than the
# synthesizer. The synthesizer's job was to write a good email. The risk
# assessor's job is to decide whether a human needs to review it before
# it gets sent. These are genuinely different questions.
#
# interrupt() is the key LangGraph primitive here. When called:
#   1. The full state is saved to the checkpointer (MemorySaver here).
#   2. The current graph.invoke() returns with an interrupt signal.
#   3. The thread is free. Nothing is blocked.
#   4. To resume: graph.invoke(Command(resume={...})) with the same thread_id.
#   5. The graph restores from the checkpointer and continues at this node.

from langgraph.types import interrupt

HITL_THRESHOLD = 100.0  # Refunds over this amount need a human to approve


def risk_assessor(state):
    """Agent 4: evaluate the draft resolution for risk.

    Returns risk_level 'low' (auto-send) or 'high' (needs human approval).
    The threshold is $100 disputed balance. Anything above that goes to HITL.
    We also flag cases where the draft makes claims the policy evidence doesn't
    clearly support.
    """
    sql = state.get("sql_result", {})
    rows = sql.get("rows", [])
    refund_amount = float(rows[0].get("disputed_balance", 0) or 0) if rows else 0.0

    prompt = f"""Assess the risk of this refund resolution draft.

Draft:
{state.get('draft_email', 'No draft.')}

Refund amount: ${refund_amount:.2f}
Threshold: ${HITL_THRESHOLD:.2f}
Policy evidence was retrieved: {bool(state.get('policy_results'))}

Mark as HIGH risk if:
- Refund exceeds the threshold, or
- The draft makes a promise not clearly supported by the policy evidence, or
- The policy application is ambiguous.

Otherwise mark LOW.

Return only valid JSON:
{{"risk_level": "low", "refund_amount": {refund_amount}, "reason": "brief explanation"}}"""

    result = call_llm_json(prompt)
    if "error" in result:
        # When in doubt, treat as high risk. Better to over-review than under-review.
        result = {"risk_level": "high", "refund_amount": refund_amount, "reason": "parse failure"}

    return {
        "risk": result,
        "messages": [AIMessage(content=f"Risk: {result.get('risk_level')} -- {result.get('reason')}")],
    }


def human_approval_node(state):
    """The HITL gate. Pauses the graph and waits for a human decision.

    interrupt() here is not a conditional skip. It genuinely pauses execution.
    The payload passed to interrupt() is what the human reviewer sees.
    The graph resumes when Command(resume={...}) is passed to graph.invoke()
    with the same thread_id.
    """
    decision = interrupt(
        {
            "message": "High-risk refund needs human approval before sending.",
            "draft_email": state.get("draft_email", ""),
            "risk_assessment": state.get("risk", {}),
            "ticket_ref": state.get("ticket_ref"),
            "query": state.get("query"),
        }
    )
    approved = decision.get("approved", False) if isinstance(decision, dict) else False
    return {
        "approved": approved,
        "messages": [AIMessage(content=f"Human decision: {'APPROVED' if approved else 'DENIED'}")],
    }


def finalize_node(state):
    """Final node: log the outcome.

    In production this would trigger the email send and update the CRM.
    Here we just record the outcome in final_status.
    """
    risk_level = (state.get("risk") or {}).get("risk_level", "low")
    approved = state.get("approved", False)

    if risk_level == "low" or approved:
        status = f"EMAIL_SENT: {state.get('draft_email', '')[:80]}..."
    else:
        status = "DENIED: Resolution held for review. Human declined approval."

    return {
        "final_status": status,
        "messages": [AIMessage(content=status[:120])],
    }


print(f"Risk assessor and HITL gate defined. Threshold: ${HITL_THRESHOLD:.2f}")

Risk assessor and HITL gate defined. Threshold: $100.00


In [5]:
# Compile the graph.
#
# The wiring here is what produces the parallel fan-out.
# route_after_triage returns a list of branch names. LangGraph runs all
# of them concurrently, then waits for all to finish before the synthesizer.
#
# MemorySaver is the checkpointer. For interrupt() to work, a checkpointer
# must be passed to compile(). In production use PostgresSaver or SqliteSaver.

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


def route_after_triage(state):
    """Decide which branches run in parallel based on what triage determined.

    Returning a list means all items in the list run concurrently.
    This is the parallel fan-out.
    """
    branches = []
    if state.get("needs_sql", True):
        branches.append("sql_agent")
    if state.get("needs_policy", False):
        branches.append("policy_agent")
    return branches or ["synthesizer_agent"]


def route_after_risk(state):
    """Route to HITL gate for high-risk resolutions, straight to finalize for low risk."""
    risk_level = (state.get("risk") or {}).get("risk_level", "low")
    return "human_approval_node" if risk_level == "high" else "finalize_node"


builder = StateGraph(AgentState)

builder.add_node("triage_agent", triage_agent)
builder.add_node("sql_agent", sql_agent)
builder.add_node("policy_agent", policy_agent)
builder.add_node("synthesizer_agent", synthesizer_agent)
builder.add_node("risk_assessor", risk_assessor)
builder.add_node("human_approval_node", human_approval_node)
builder.add_node("finalize_node", finalize_node)

builder.add_edge(START, "triage_agent")

# Fan-out: triage decides which branches run, both run in parallel
builder.add_conditional_edges(
    "triage_agent",
    route_after_triage,
    {
        "sql_agent": "sql_agent",
        "policy_agent": "policy_agent",
        "synthesizer_agent": "synthesizer_agent",
    },
)

# Fan-in: both branches converge on the synthesizer
builder.add_edge("sql_agent", "synthesizer_agent")
builder.add_edge("policy_agent", "synthesizer_agent")

builder.add_edge("synthesizer_agent", "risk_assessor")
builder.add_conditional_edges(
    "risk_assessor",
    route_after_risk,
    {"human_approval_node": "human_approval_node", "finalize_node": "finalize_node"},
)
builder.add_edge("human_approval_node", "finalize_node")
builder.add_edge("finalize_node", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("Graph compiled with MemorySaver checkpointer.")
print("Nodes:", list(graph.nodes))

Graph compiled with MemorySaver checkpointer.
Nodes: ['__start__', 'triage_agent', 'sql_agent', 'policy_agent', 'synthesizer_agent', 'risk_assessor', 'human_approval_node', 'finalize_node']


In [6]:
# Test the graph with three scenarios.
#
# Test A: Low-value ticket ($45). Should auto-resolve without pausing.
# Test B: High-value ticket ($275). Should pause at the HITL gate.
#         Then resume with approval and confirm the email is sent.
# Test C: Same high-value ticket, but resume with denial.

from langgraph.types import Command


def make_initial_state(query, user_id="dave", role="support_agent", ticket_ref=None):
    return {
        "messages": [HumanMessage(content=query)],
        "query": query,
        "token": "dave-mock-jwt",
        "role": role,
        "user_id": user_id,
        "token_valid": True,
        "needs_sql": False,
        "needs_policy": False,
        "ticket_ref": ticket_ref,
        "sql_result": {},
        "policy_results": [],
        "draft_email": "",
        "risk": {},
        "approved": False,
        "final_status": "",
        "thread_id": str(uuid.uuid4()),
    }


def run_graph(initial_state, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(initial_state, config=config)
    snapshot = graph.get_state(config)
    interrupted = bool(snapshot.next)
    return result, interrupted


print("--- Test A: Low-value ticket (T1001, $45) ---")
thread_a = str(uuid.uuid4())
state_a = make_initial_state(
    query="Alice Smith on ticket T1001 says her $45 refund has not arrived in 30 days. Please resolve.",
    ticket_ref="T1001",
)
try:
    result_a, interrupted_a = run_graph(state_a, thread_a)
    print(f"Interrupted: {interrupted_a}")
    print(f"Risk level: {result_a.get('risk', {}).get('risk_level', 'N/A')}")
    print(f"Final status: {result_a.get('final_status', '')[:120]}")
    assert not interrupted_a, "Expected auto-resolve for low-value ticket."
    assert "EMAIL_SENT" in result_a.get("final_status", ""), "Expected EMAIL_SENT."
    print("PASS")
except Exception as e:
    print(f"Test A error: {e}")


print("\n--- Test B: High-value ticket (T1010, $275) -- expects HITL pause ---")
thread_b = str(uuid.uuid4())
state_b = make_initial_state(
    query="Carol White on ticket T1010 has a $275 disputed balance on damaged goods. Please resolve.",
    ticket_ref="T1010",
)
try:
    result_b, interrupted_b = run_graph(state_b, thread_b)
    print(f"Interrupted: {interrupted_b}")
    print(f"Risk level: {result_b.get('risk', {}).get('risk_level', 'N/A')}")

    if interrupted_b:
        config_b = {"configurable": {"thread_id": thread_b}}
        paused = graph.get_state(config_b)
        print(f"Graph paused at: {paused.next}")
        print(f"Draft email snippet: {paused.values.get('draft_email', '')[:150]}")

        print("\nResuming with APPROVAL...")
        resumed = graph.invoke(Command(resume={"approved": True}), config=config_b)
        print(f"Final status: {resumed.get('final_status', '')[:120]}")
        assert "EMAIL_SENT" in resumed.get("final_status", ""), "Expected EMAIL_SENT after approval."
        print("PASS")
    else:
        print("Graph did not pause. Check the risk_assessor threshold logic.")
except Exception as e:
    print(f"Test B error: {e}")


print("\n--- Test C: High-value ticket -- resume with DENIAL ---")
thread_c = str(uuid.uuid4())
state_c = make_initial_state(
    query="T1010: Carol White is asking for a $275 refund on disputed damaged goods.",
    ticket_ref="T1010",
)
try:
    result_c, interrupted_c = run_graph(state_c, thread_c)
    if interrupted_c:
        config_c = {"configurable": {"thread_id": thread_c}}
        denied = graph.invoke(Command(resume={"approved": False}), config=config_c)
        print(f"Final status: {denied.get('final_status', '')[:120]}")
        assert "DENIED" in denied.get("final_status", ""), "Expected DENIED."
        print("PASS")
    else:
        print("Graph did not pause for this test (risk was assessed as low).")
except Exception as e:
    print(f"Test C error: {e}")

--- Test A: Low-value ticket (T1001, $45) ---


Interrupted: False
Risk level: low
Final status: EMAIL_SENT: Subject: Update on Ticket T1001

Dear Alice Smith,

Thank you for your patience ...
PASS

--- Test B: High-value ticket (T1010, $275) -- expects HITL pause ---
Interrupted: True
Risk level: high
Graph paused at: ('human_approval_node',)
Draft email snippet: Subject: Update on Ticket T1010

Dear Carol,

I am writing regarding your escalated ticket T1010 concerning a $275.00 disputed balance for damaged goo

Resuming with APPROVAL...
Final status: EMAIL_SENT: Subject: Update on Ticket T1010

Dear Carol,

I am writing regarding your escala...
PASS

--- Test C: High-value ticket -- resume with DENIAL ---
Final status: DENIED: Resolution held for review. Human declined approval.
PASS
